# MassMIND Segmentation — Colab Training

Runs the U-Net + VGG-16 baseline on a Colab GPU for each of the three augmentation pipelines (A, B, C) and produces a comparison of their training curves and final per-class IoU.

**Prerequisites**

1. **GPU runtime**: Runtime → Change runtime type → T4 GPU (or better).
2. **Private repo access**: have a GitHub Personal Access Token with `repo` scope ready (github.com → Settings → Developer Settings → Personal access tokens). You'll be prompted for it in the clone cell — it never gets written to disk or the notebook output.
3. **(Optional) Google Drive**: if you want checkpoints to survive the session ending, run the optional Drive-mount cell below.

**Expected runtime**: ~30–60 min per config on a T4 (30 epochs × 2042 images). All three configs back-to-back fit comfortably inside Colab's free 12-hour session limit.

## 0. Config

Edit this cell to change which configs to run, epochs, batch size, etc.

In [ ]:
REPO_URL_HTTPS = 'https://github.com/whothemann/massmind-segmentation.git'  # <-- EDIT ME
REPO_BRANCH = 'main'
REPO_DIR = 'massmind-segmentation'

# Training
EPOCHS = 30
BATCH_SIZE = 8
LR = 1e-4
AUGMENTATIONS = ['A', 'B', 'C']  # subset to speed up (e.g. ['A'])

# Persistence (optional). Set to None to keep everything in the ephemeral runtime.
DRIVE_RUNS_DIR = None   # e.g. '/content/drive/MyDrive/massmind_runs'
DRIVE_HF_CACHE = None   # e.g. '/content/drive/MyDrive/hf_cache' -- avoids re-downloading VGG-16 weights

## 1. Clone the repository

You'll be prompted for a GitHub Personal Access Token. The token is read via `getpass` so it does not get echoed or saved into the notebook output.

In [ ]:
import os
import getpass
import subprocess
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f'{REPO_DIR} already present — pulling latest from {REPO_BRANCH}')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    if not token:
        raise RuntimeError('No token provided.')
    # Inject the token into the HTTPS URL only for this single git operation.
    auth_url = REPO_URL_HTTPS.replace('https://', f'https://x-access-token:{token}@')
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, auth_url, REPO_DIR], check=True)
    # Rewrite the remote without the token so it isn't stored in .git/config.
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL_HTTPS], check=True)
    del token, auth_url

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 2. Install dependencies

Colab T4 ships with a recent torch + CUDA stack pre-installed, so this step mostly pulls albumentations / smp / gdown / opencv-headless. ~1–2 minutes.

In [ ]:
!pip install -q -r requirements.txt

## 3. GPU sanity check

In [ ]:
import torch
print('torch        :', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device       :', torch.cuda.get_device_name(0))
    print('vram (GB)    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

## 4. (Optional) Mount Google Drive for persistence

If `DRIVE_RUNS_DIR` is set in the config cell, run this to mount Drive and redirect run outputs + HuggingFace cache there. Otherwise everything lives in the ephemeral Colab runtime — fine for one session, gone on disconnect.

In [ ]:
if DRIVE_RUNS_DIR or DRIVE_HF_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    if DRIVE_RUNS_DIR:
        Path(DRIVE_RUNS_DIR).mkdir(parents=True, exist_ok=True)
        print('Run outputs will be saved under:', DRIVE_RUNS_DIR)
    if DRIVE_HF_CACHE:
        Path(DRIVE_HF_CACHE).mkdir(parents=True, exist_ok=True)
        os.environ['HF_HOME'] = DRIVE_HF_CACHE
        print('HF cache redirected to:', DRIVE_HF_CACHE)
else:
    print('No Drive paths set — running ephemerally.')

## 5. Download dataset and prepare splits + stats

Idempotent — re-running just verifies the files are present. ~1–2 min the first time (Google Drive download).

In [ ]:
!python scripts/download.py
!python -m src.splits
!python -m src.stats

## 6. Train

Runs are split into three separate cells (A, B, C) so you can monitor each one independently and re-run a single config without redoing the others. The helper cell below defines `run_training(aug)`, which:

- streams the trainer's stdout line by line,
- shows a `tqdm` progress bar with the current / total epoch,
- updates a live loss + mIoU plot at the end of each epoch.

Each run dumps `runs/unet_vgg16_aug<X>_<ts>/` with checkpoints, `metrics.csv`, and `config.json`. If `DRIVE_RUNS_DIR` is set, the run directory is also copied to Drive after the run completes.

In [ ]:
import re
import shutil
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.notebook import tqdm

# Will be populated by the per-augmentation cells below.
run_dirs = {}

# Matches the per-epoch log line emitted by src.train, e.g.:
#   epoch 12/30  train_loss=0.4321  val_loss=0.3987  mIoU=0.6541  pixel_acc=0.8123  (45.3s)
_EPOCH_RE = re.compile(
    r"epoch\s+(\d+)/(\d+)\s+train_loss=([\d.eE+-]+)\s+val_loss=([\d.eE+-]+)\s+mIoU=([\d.eE+-]+)"
)

# Colab free T4 only has 2 vCPUs; the trainer's CUDA default is 4 and
# oversubscribes, slowing dataloading. Match it to the actual core count.
NUM_WORKERS = 2


def run_training(aug):
    """Launch src.train for one augmentation with a live progress bar + plot.

    Returns the output directory path on success, or raises on non-zero exit.
    """
    ts = int(time.time())
    out_dir = f"runs/unet_vgg16_aug{aug}_{ts}"
    print(f"=== Training augmentation {aug} -> {out_dir} ===")

    cmd = [
        "python", "-u", "-m", "src.train",
        "--augmentation", aug,
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--num-workers", str(NUM_WORKERS),
        "--output-dir", out_dir,
    ]

    pbar = tqdm(total=EPOCHS, desc=f"aug {aug}", unit="epoch")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    plot_handle = display(fig, display_id=True)
    epochs, tloss, vloss, mious = [], [], [], []

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            print(line)
            m = _EPOCH_RE.search(line)
            if not m:
                continue
            ep, total = int(m.group(1)), int(m.group(2))
            tl, vl, mi = float(m.group(3)), float(m.group(4)), float(m.group(5))
            epochs.append(ep); tloss.append(tl); vloss.append(vl); mious.append(mi)
            pbar.total = total
            pbar.update(ep - pbar.n)

            axes[0].clear(); axes[1].clear()
            axes[0].plot(epochs, tloss, label="train", marker="o", markersize=3)
            axes[0].plot(epochs, vloss, label="val",   marker="o", markersize=3)
            axes[0].set_title(f"aug {aug} -- loss"); axes[0].set_xlabel("epoch")
            axes[0].set_ylabel("CE loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
            axes[1].plot(epochs, mious, color="C2", marker="o", markersize=3)
            axes[1].set_title(f"aug {aug} -- val mIoU"); axes[1].set_xlabel("epoch")
            axes[1].set_ylabel("mIoU"); axes[1].grid(alpha=0.3)
            fig.tight_layout()
            plot_handle.update(fig)
    except KeyboardInterrupt:
        proc.terminate()
        raise
    finally:
        proc.wait()
        pbar.close()
        plt.close(fig)

    if proc.returncode != 0:
        raise RuntimeError(f"Training for aug {aug} exited with code {proc.returncode}")

    if DRIVE_RUNS_DIR:
        drive_path = Path(DRIVE_RUNS_DIR) / Path(out_dir).name
        shutil.copytree(out_dir, drive_path, dirs_exist_ok=True)
        print(f"Copied to Drive: {drive_path}")

    return out_dir

### 6a. Augmentation A

Pipeline A: geometric augmentations only (random flip, scale, crop). Train and watch progress.

In [ ]:
run_dirs['A'] = run_training('A')

### 6b. Augmentation B

Pipeline B: geometric + photometric augmentations. Train and watch progress.

In [ ]:
run_dirs['B'] = run_training('B')

### 6c. Augmentation C

Pipeline C: minimal augmentations (normalization only). Train and watch progress.

In [ ]:
run_dirs['C'] = run_training('C')

## 7. Compare runs

Loads `metrics.csv` for each completed run and overlays train/val loss + mIoU curves, plus a final per-class IoU table.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

metrics = {aug: pd.read_csv(Path(d) / 'metrics.csv') for aug, d in run_dirs.items()}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for aug, df in metrics.items():
    axes[0].plot(df['epoch'], df['train_loss'], label=f'{aug} train', linestyle='--')
    axes[0].plot(df['epoch'], df['val_loss'],   label=f'{aug} val')
    axes[1].plot(df['epoch'], df['mIoU'],       label=f'{aug}')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('CE loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mIoU'); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
CLASS_NAMES = ['sky', 'water', 'bridge', 'obstacle', 'living_obs', 'background', 'self']
rows = []
for aug, df in metrics.items():
    # Pick the row with best val mIoU.
    best = df.loc[df['mIoU'].idxmax()]
    row = {'aug': aug, 'best_epoch': int(best['epoch']), 'mIoU': best['mIoU'], 'pixel_acc': best['pixel_acc']}
    for n in CLASS_NAMES:
        row[f'iou_{n}'] = best[f'iou_{n}']
    rows.append(row)
summary = pd.DataFrame(rows).set_index('aug')
summary

## 8. Push run outputs back to the repo (optional)

If you want `metrics.csv` + `config.json` in the repo for reproducibility (without committing the heavy `.pt` checkpoints), uncomment and run. The `.gitignore` already excludes `runs/`, so we have to add the metrics files explicitly with `git add -f`.

In [ ]:
# COMMIT_RESULTS = False
# if COMMIT_RESULTS:
#     for aug, d in run_dirs.items():
#         !git add -f {d}/metrics.csv {d}/config.json
#     !git -c user.email='colab@local' -c user.name='colab' commit -m 'add training metrics from Colab run'
#     # You will need to push manually with a PAT — see the clone cell pattern.